In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os

os.chdir('..')

CSV_DIR = 'data/csv'
IMG_DIR = 'data/Pharyngitis_dataset'

In [2]:
from pathlib import Path

positive_dir_path = Path(os.path.join(IMG_DIR,'train/phar'))
pos_img_names = []

for file_path in positive_dir_path.iterdir():
    if file_path.is_file() and file_path.suffix.lower() in ['.png', '.jpg', '.jpeg', '.gif']:
        pos_img_names.append(os.path.join("train/phar",file_path.name)) # Use .name to get only the filename


negative_dir_path = Path(os.path.join(IMG_DIR,'train/no'))
neg_img_names = []

for file_path in negative_dir_path.iterdir():
    if file_path.is_file() and file_path.suffix.lower() in ['.png', '.jpg', '.jpeg', '.gif']:
        neg_img_names.append(os.path.join("train/no",file_path.name)) # Use .name to get only the filename

In [3]:
df_pos = pd.DataFrame(pos_img_names, columns=['ImageName'])
df_pos.reset_index(drop=True, inplace=True)
df_pos['Labels'] = 1
df_neg = pd.DataFrame(neg_img_names, columns=['ImageName'])
df_pos.reset_index(drop=True, inplace=True)
df_neg['Labels'] = 0

df = pd.concat([df_pos, df_neg])
df.to_csv(os.path.join(CSV_DIR, 'new_data.csv'), index=False)

labels = df.iloc[:, 1]
indices = range(len(df))
train_idx, test_idx = train_test_split(
    indices, 
    test_size=0.3, 
    stratify=labels, 
    random_state=42
)
print(len(df))

329


In [4]:
class StrepDataset(Dataset):
    def __init__(self, csv, img_dir, transform = None):
        self.data = pd.read_csv(csv)
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.data.iloc[idx, 0])
        image = Image.open(img_name).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(self.data.iloc[idx, 1])

        return image, label

In [5]:
class StrepClassifier(nn.Module):
    def __init__(self, num_features, num_classes=1):
        super(StrepClassifier, self).__init__()
        
        # 1. Image Branch: Pre-trained ResNet18
        # remove the final fully connected layer to get visual embeddings
        resnet = models.resnet18(weights='DEFAULT')

        for param in resnet.parameters():
            param.requires_grad = False
        
        for param in resnet.layer4.parameters():
            param.requires_grad = True
        
        self.image_branch = nn.Sequential(*list(resnet.children())[:-1])

        self.img_projection = nn.Sequential(
            nn.Linear(512, 32),
            nn.BatchNorm1d(32), # Forces features to be learnable
            nn.ReLU()
        )
        
        # # # 3. Fusion Layer: Combines image embeddings (512) + feature embeddings (32)
        self.classifier = nn.Sequential(
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(16, num_classes), # Output for 2 classes
        )

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, images):
        # Process image
        img_out = self.image_branch(images)
        img_out = torch.flatten(img_out, 1) # Shape: [batch, 512]
        img_out = self.img_projection(img_out)

        return self.classifier(img_out)

In [6]:
def white_balance(img):

    img = np.array(img).astype(float)

    ## average value of each channel
    avg_red = np.mean(img[:, :, 0])
    avg_green = np.mean(img[:, :, 1])
    avg_blue = np.mean(img[:, :, 2])
    
    # gray value calculation
    avg_gray = (avg_red + avg_green + avg_blue) / 3
    
    # Scaling each channel
    img[:, :, 0] *= (avg_gray / avg_red)
    img[:, :, 1] *= (avg_gray / avg_green)
    img[:, :, 2] *= (avg_gray / avg_blue)
    
    # Clip values to [0, 255]
    img = np.clip(img, 0, 255).astype(np.uint8)     #the factor can cause values to go beyond 255, hence bounding it.
    return Image.fromarray(img)

def apply_clahe(img):
    img_np = np.array(img)
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    lab = cv2.merge((l,a,b))
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

In [8]:
crop_size = 160
resize_img_dim = (224, 224)

transforms_train = transforms.Compose([
    transforms.Lambda(lambda x: white_balance(x)),
    transforms.Lambda(lambda x: apply_clahe(x)),
    transforms.RandomResizedCrop(resize_img_dim, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomRotation(degrees=20),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transforms_test = transforms.Compose([
    transforms.Lambda(lambda x: white_balance(x)),       ## Fix the blur
    transforms.Lambda(lambda x: apply_clahe(x)),         ## Brighten dark areas
    transforms.Resize(resize_img_dim),
    transforms.CenterCrop(crop_size),                    ## Crop to focus only on the tonsil area, disregard tongue / teeth
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Created datasets for the train and test data with individual transforms
train_dataset = StrepDataset(csv=os.path.join(CSV_DIR, 'new_data.csv'), img_dir= IMG_DIR, transform=transforms_train)
test_dataset = StrepDataset(csv=os.path.join(CSV_DIR, 'new_data.csv'), img_dir= IMG_DIR, transform=transforms_test)

## Created Subsets for the train and test dataset indices
train_subset = Subset(train_dataset, train_idx)
test_subset = Subset(test_dataset, test_idx)

## Created Dataloaders for the train and test dataset 
train_loader = DataLoader(train_subset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=16, shuffle=True)


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = StrepClassifier(num_features = 7, num_classes = 1).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001, weight_decay=1e-2)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-2)
epochs = 50

for epoch in range(epochs):
    model.train()
    
    train_loss = 0.0
    preds_train = 0.0
    correct_train = 0.0
    total_train = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device),  labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(images).view(-1) ## [batch_size, 1] -> [batch_size]
        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        train_loss += loss.item()

        preds_train = (outputs > 0).float()
        correct_train += (preds_train == labels).sum().item()
        total_train += labels.size(0)

    train_accuracy = 100 * correct_train / total_train
    print("Epoch:", epoch)
    print("Train loss:", train_loss)
    print(f"Train Acc: {train_accuracy:.2f}%")

    with torch.no_grad(): # Disable gradient calculation to save memory/speed
        val_loss = 0.0
        correct = 0.0
        total = 0.0
        model.eval()
        for images,  labels in test_loader:
            images,  labels = images.to(device),labels.to(device).float()
            
            outputs = model(images).view(-1)
            val_loss += criterion(outputs, labels).item()
            
            # Convert probabilities to binary predictions (0 or 1)
            preds = (outputs > 0).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    # Print progress
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(test_loader)
    accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Val Acc: {accuracy:.2f}%")
    print()
        

Epoch: 0
Train loss: 12.622372031211853
Train Acc: 48.70%
Epoch [1/50] Train Loss: 0.8415 | Val Loss: 1.0978 | Val Acc: 49.49%

Epoch: 1
Train loss: 11.343636810779572
Train Acc: 58.70%
Epoch [2/50] Train Loss: 0.7562 | Val Loss: 1.0523 | Val Acc: 52.53%

Epoch: 2
Train loss: 10.376440346240997
Train Acc: 63.48%
Epoch [3/50] Train Loss: 0.6918 | Val Loss: 0.7400 | Val Acc: 66.67%

Epoch: 3
Train loss: 9.476355642080307
Train Acc: 64.78%
Epoch [4/50] Train Loss: 0.6318 | Val Loss: 0.7237 | Val Acc: 69.70%

Epoch: 4
Train loss: 8.309980481863022
Train Acc: 69.13%
Epoch [5/50] Train Loss: 0.5540 | Val Loss: 0.5712 | Val Acc: 68.69%

Epoch: 5
Train loss: 7.761338621377945
Train Acc: 74.35%
Epoch [6/50] Train Loss: 0.5174 | Val Loss: 0.5306 | Val Acc: 80.81%

Epoch: 6
Train loss: 6.6015748381614685
Train Acc: 80.87%
Epoch [7/50] Train Loss: 0.4401 | Val Loss: 0.5728 | Val Acc: 76.77%

Epoch: 7
Train loss: 6.5507572889328
Train Acc: 81.30%
Epoch [8/50] Train Loss: 0.4367 | Val Loss: 0.5248 |

In [ ]:
from pathlib import Path

test_dir_path = Path("Pharyngitis_dataset/test")
test_img_names = []
test_label_names = []

for file_path in test_dir_path.iterdir():
    if file_path.is_file() and file_path.suffix.lower() in ['.png', '.jpg', '.jpeg', '.gif']:
        label = 1 
        if file_path.name[0] == 'n':
            label = 0
        test_img_names.append(os.path.join("test",file_path.name)) # Use .name to get only the filename
        test_label_names.append(label)

print(test_img_names)

['test\\no_pharyngitis (1).JPG', 'test\\no_pharyngitis (10).JPG', 'test\\no_pharyngitis (11).JPG', 'test\\no_pharyngitis (12).JPG', 'test\\no_pharyngitis (13).JPG', 'test\\no_pharyngitis (14).JPG', 'test\\no_pharyngitis (15).JPG', 'test\\no_pharyngitis (16).JPG', 'test\\no_pharyngitis (17).JPG', 'test\\no_pharyngitis (18).JPG', 'test\\no_pharyngitis (19).JPG', 'test\\no_pharyngitis (2).JPG', 'test\\no_pharyngitis (20).JPG', 'test\\no_pharyngitis (3).JPG', 'test\\no_pharyngitis (4).JPG', 'test\\no_pharyngitis (5).JPG', 'test\\no_pharyngitis (6).JPG', 'test\\no_pharyngitis (7).JPG', 'test\\no_pharyngitis (8).JPG', 'test\\no_pharyngitis (9).JPG', 'test\\pharyngitis (1).JPG', 'test\\pharyngitis (10).JPG', 'test\\pharyngitis (11).JPG', 'test\\pharyngitis (12).JPG', 'test\\pharyngitis (13).JPG', 'test\\pharyngitis (2).JPG', 'test\\pharyngitis (3).JPG', 'test\\pharyngitis (4).JPG', 'test\\pharyngitis (5).JPG', 'test\\pharyngitis (6).JPG', 'test\\pharyngitis (7).JPG', 'test\\pharyngitis (8).JP

In [ ]:
df_test = pd.DataFrame(test_img_names, columns=['ImageName'])
df_test_labels = pd.DataFrame(test_label_names, columns=['label'])

df_test.reset_index(drop=True, inplace=True)
df_test_labels.reset_index(drop=True, inplace=True)


df = pd.concat([df_test, df_test_labels], axis=1)
df.to_csv(os.path.join(CSV_DIR,'test_data.csv'), index=False)

print(len(df))

33


In [ ]:

test_dataset = StrepDataset(csv='test_data.csv', img_dir=IMG_DIR, transform=transforms_test)

test_subset = Subset(test_dataset, range(len(df)))

test_loader = DataLoader(test_subset, batch_size=16, shuffle=True)

In [ ]:
with torch.no_grad(): # Disable gradient calculation to save memory/speed
        val_loss = 0.0
        correct = 0.0
        total = 0.0
        model.eval()
        for images,  labels in test_loader:
                images,  labels = images.to(device),labels.to(device).float()
                
                outputs = model(images).view(-1)
                val_loss += criterion(outputs, labels).item()
                
                # Convert probabilities to binary predictions (0 or 1)
                preds = (outputs > 0).float()
                correct += (preds == labels).sum().item()
                total += labels.size(0)

avg_val_loss = val_loss / len(test_loader)
accuracy = 100 * correct / total

print(f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {accuracy:.2f}%")
print()

Epoch [50/50] Train Loss: 0.2812 | Val Loss: 0.2897 | Val Acc: 84.85%

